# SFDI-Net Workflow

Notebook workflow for training and inference with SFDI-Net:

1. Imports
2. Configuration
3. Data Preparation
4. Training
5. Inference


## 1. Imports

In [ ]:
from pathlib import Path
import sys
import json

import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'dual_branch_inpainting').exists():
    ROOT = ROOT.parent
if not (ROOT / 'dual_branch_inpainting').exists():
    raise FileNotFoundError('Could not locate the project root containing dual_branch_inpainting/')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_ROOT = ROOT / 'data'
OUTPUT_ROOT = ROOT / 'outputs' / 'sfdi_net'

from dual_branch_inpainting.workflow import train_main, infer_main
from dual_branch_inpainting.data.datasets import ImageMaskPairDataset


## 2. Configuration

In [ ]:
# Configure training and inference parameters
cfg = {
    'seed': 0,
    'device_index': 0,
    'in_channels': 3,
    'base_ch': 64,
    'num_levels': 6,
    'max_channels': 512,
    'fsj_levels': '1,2,3,4',

    # Path settings
    'output_root': str(OUTPUT_ROOT),
    'pretrained_unet_path': '',
    'images_dir': str(DATA_ROOT / 'train_images_with_gaps'),
    'masks_dir': str(DATA_ROOT / 'train_masks'),
    'test_images_dir': str(DATA_ROOT / 'test_images'),
    'test_masks_dir': str(DATA_ROOT / 'test_masks'),

    # Image crop and size settings
    'target_width': 512,
    'crop_height': 256,
    'final_width': 512,
    'overlap': 16,

    # Training optimization settings
    'batch_size': 3,
    'num_epochs': 30,
    'g_lr': 1e-5,
    'd_lr': 1e-5,
    'lambda_l1': 6,
    'lambda_perc': 1.0,
    'lambda_style': 35.0,
    'lambda_tv': 0.1,
    'lambda_grad': 30.0,
    'lambda_ms_ssim': 15.0,
    'lambda_color': 6.0,
    'num_workers': 4,
    'prefetch_factor': 2,

    # GAN settings
    'lambda_gan': 1,
    'w_global': 1.0,
    'w_local': 2.0,
    'gan_type': 'lsgan',
    'd_steps': 1,

    # Synthetic mask settings used during training
    'syn_min_stripe': 20,
    'syn_max_stripe': 50,
    'syn_min_gap': 20,
    'syn_max_gap': 60,
    'syn_coverage_min': 0.1,
    'syn_coverage_max': 0.4,
    'syn_skew_min': -20,
    'syn_skew_max': 20,

    # Inference settings
    'infer_checkpoint_path': str(OUTPUT_ROOT / 'checkpoints' / 'latest.pt'),
    'infer_dir': str(OUTPUT_ROOT / 'infer'),
    'output_suffix': 'sfdi_net',

    # Training flow control
    'no_resume': True,
}

print(json.dumps(cfg, indent=2, ensure_ascii=False))


## 3. Data Preparation

In [ ]:
# Build the paired training dataset and inspect one sample
dataset = ImageMaskPairDataset(
    images_dir=cfg['images_dir'],
    masks_dir=cfg['masks_dir'],
    target_width=cfg['target_width'],
    crop_height=cfg['crop_height'],
    final_width=cfg['final_width'],
    overlap=cfg['overlap'],
    use_grayscale=False,
    enable_cache=True,
)

sample = dataset[0]
print('Dataset size:', len(dataset))
print('Image tensor shape:', tuple(sample['image'].shape))
print('Mask tensor shape:', tuple(sample['natural_mask'].shape))


## 4. Training

In [ ]:
def build_train_argv(config: dict):
    """Convert all training options into CLI arguments."""
    argv = [
        '--seed', str(config['seed']),
        '--device-index', str(config['device_index']),
        '--in-channels', str(config['in_channels']),
        '--base-ch', str(config['base_ch']),
        '--num-levels', str(config['num_levels']),
        '--max-channels', str(config['max_channels']),
        '--fsj-levels', config['fsj_levels'],
        '--images-dir', config['images_dir'],
        '--masks-dir', config['masks_dir'],
        '--output-root', config['output_root'],
        '--target-width', str(config['target_width']),
        '--crop-height', str(config['crop_height']),
        '--final-width', str(config['final_width']),
        '--overlap', str(config['overlap']),
        '--batch-size', str(config['batch_size']),
        '--num-epochs', str(config['num_epochs']),
        '--g-lr', str(config['g_lr']),
        '--d-lr', str(config['d_lr']),
        '--lambda-l1', str(config['lambda_l1']),
        '--lambda-perc', str(config['lambda_perc']),
        '--lambda-style', str(config['lambda_style']),
        '--lambda-tv', str(config['lambda_tv']),
        '--lambda-grad', str(config['lambda_grad']),
        '--lambda-ms-ssim', str(config['lambda_ms_ssim']),
        '--lambda-color', str(config['lambda_color']),
        '--num-workers', str(config['num_workers']),
        '--prefetch-factor', str(config['prefetch_factor']),
        '--lambda-gan', str(config['lambda_gan']),
        '--w-global', str(config['w_global']),
        '--w-local', str(config['w_local']),
        '--gan-type', config['gan_type'],
        '--d-steps', str(config['d_steps']),
        '--syn-min-stripe', str(config['syn_min_stripe']),
        '--syn-max-stripe', str(config['syn_max_stripe']),
        '--syn-min-gap', str(config['syn_min_gap']),
        '--syn-max-gap', str(config['syn_max_gap']),
        '--syn-coverage-min', str(config['syn_coverage_min']),
        '--syn-coverage-max', str(config['syn_coverage_max']),
        '--syn-skew-min', str(config['syn_skew_min']),
        '--syn-skew-max', str(config['syn_skew_max']),
    ]
    if config.get('pretrained_unet_path'):
        argv.extend(['--pretrained-unet-path', config['pretrained_unet_path']])
    if config.get('no_resume', False):
        argv.append('--no-resume')
    return argv

train_argv = build_train_argv(cfg)
train_argv


In [ ]:
# Run training
train_main(train_argv)

## 5. Inference

In [ ]:
def build_infer_argv(config: dict):
    """Convert all inference options into CLI arguments."""
    return [
        '--seed', str(config['seed']),
        '--device-index', str(config['device_index']),
        '--in-channels', str(config['in_channels']),
        '--base-ch', str(config['base_ch']),
        '--num-levels', str(config['num_levels']),
        '--max-channels', str(config['max_channels']),
        '--fsj-levels', config['fsj_levels'],
        '--checkpoint-path', config['infer_checkpoint_path'],
        '--test-images-dir', config['test_images_dir'],
        '--test-masks-dir', config['test_masks_dir'],
        '--infer-dir', config['infer_dir'],
        '--target-width', str(config['target_width']),
        '--crop-height', str(config['crop_height']),
        '--final-width', str(config['final_width']),
        '--overlap', str(config['overlap']),
        '--output-suffix', config['output_suffix'],
    ]

infer_argv = build_infer_argv(cfg)
infer_argv

In [ ]:
# Run inference
infer_main(infer_argv)
